# 센텐스피스
- 내부 단어 분리를 위한 유용한 패키지
- 센텐스피스는 사전 토큰화 작업없이 단어 분리 토큰화를 수행하므로 언어에 종속되지 않습니다.
논문: https://arxiv.org/pdf/1808.06226.pdf
센텐스피스 깃허브: https://github.com/google/sentencepiece

In [1]:
!pip install sentencepiece

# IMDB 리뷰 토큰화하기

In [2]:
import sentencepiece as spm
import pandas as pd
import urllib.request
import csv

In [3]:
# IMDB 리뷰데이터를 다운 후 이를 데이터 프레임에 저장 
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/LawrenceDuan/IMDb-Review-Analysis/master/IMDb_Reviews.csv",
    filename="IMDb_Reviews.csv"
)


('IMDb_Reviews.csv', <http.client.HTTPMessage at 0x29fa7e73a90>)

In [4]:
train_df = pd.read_csv('IMDb_Reviews.csv')
train_df['review']

0        My family and I normally do not watch local mo...
1        Believe it or not, this was at one time the wo...
2        After some internet surfing, I found the "Home...
3        One of the most unheralded great works of anim...
4        It was the Sixties, and anyone with long hair ...
                               ...                        
49995    the people who came up with this are SICK AND ...
49996    The script is so so laughable... this in turn,...
49997    "So there's this bride, you see, and she gets ...
49998    Your mind will not be satisfied by this nobud...
49999    The chaser's war on everything is a weekly sho...
Name: review, Length: 50000, dtype: object

In [5]:
print('리뷰 개수 :', len(train_df))  # 리뷰 개수 출력

리뷰 개수 : 50000


In [6]:
# 센텐스피스 입력으로 사용하기 위해 데이터프레임 , txt파일로 저장
with open('imdb_review.txt', 'w', encoding='utf8') as f:
    f.write('\n'.join(train_df['review']))

In [ ]:
# 센텐스피스로 단어 집합과 각 단어에 고유한 정수 부여
# BPE(Byte Pair Encoding)
spm.SentencePieceTrainer.Train(
    '--input=imdb_review.txt '
    '--model_prefix=imdb '
    '--vocab_size=5000 '
    '--model_type=bpe '
    '--max_sentence_length=9999'
)

In [8]:
# vocab 생성이 완료되면 imdb.model, imdb.vocab 파일 두개가 생성 
# 단어 집합의 크기를 확인하기 위해 vocab 파일을 데이터프레임에 저장
vocab_list = pd.read_csv(
    'imdb.vocab',
    sep='\t',
    header=None,
    quoting=csv.QUOTE_NONE
)

vocab_list.sample(10)

,0,1
3502,▁historical,-3499
2127,▁intellig,-2124
757,▁world,-754
4462,▁hurt,-4459
3610,▁alien,-3607
1521,▁totally,-1518
1274,/10,-1271
57,il,-54
686,▁why,-683
1894,rit,-1891


In [9]:
# 단어 집합크기
len(vocab_list)

5000

In [10]:
# model 파일을 로드
sp = spm.SentencePieceProcessor()

vocab_file = "imdb.model"
sp.load(vocab_file)

True

In [11]:
# SentencePiece 토크나이저 테스트 코드
# 테스트할 문장 리스트
lines = [
    "I didn't at all think of it this way.",
    "I have waited a long time for someone to film"
]
# 각 문장에 대해 SentencePiece 토큰화 결과 확인
for line in lines:
    print(line) # 원문 출력
    # 문장을 서브워드 단위로 분리한 토큰 출력
    print(sp.encode_as_pieces(line))
    # 서브워드 토큰을 정수 ID로 변환한 결과 출력
    print(sp.encode_as_ids(line))
    # 문장 구분용 빈 줄
    print()

I didn't at all think of it this way.
['▁I', '▁didn', "'", 't', '▁at', '▁all', '▁think', '▁of', '▁it', '▁this', '▁way', '.']
[41, 624, 4950, 4926, 139, 170, 378, 30, 58, 73, 413, 4945]

I have waited a long time for someone to film
['▁I', '▁have', '▁wa', 'ited', '▁a', '▁long', '▁time', '▁for', '▁someone', '▁to', '▁film']
[41, 142, 1364, 1121, 4, 668, 285, 93, 1079, 33, 91]



In [12]:
# 단어 집합크기 확인
sp.GetPieceSize()

5000

In [13]:
# 정수로 부터 매핑되는 서브 워드로 변환
sp.IdToPiece(430)

'▁character'

In [14]:
# 서브워드로 부터 맵핑되는 정수로 변환
sp.PieceToId('▁character')

430

In [15]:
#  정수 시퀀스로부터 문장으로 변환합니다.
sp.DecodeIds([41, 141, 1364, 1120, 4, 666, 285, 92, 1078, 33, 91])

'Iul wa fall aold timeooland to film'

In [16]:
# 서브워드 시퀀스부터 문장으로 변환
sp.DecodePieces(['▁I', '▁have', '▁wa', 'ited', '▁a', '▁long', '▁time', '▁for', '▁ someone', '▁to', '▁film'])

'I have waited a long time for▁ someone to film'

In [17]:
# SentencePiece로 문장을 서브워드 토큰으로 인코딩 (문자 형태)
print(sp.encode(
    'I have waited a long time for someone to film',
    out_type=str
))

# SentencePiece로 문장을 토큰 ID(정수)로 인코딩
print(sp.encode(
    'I have waited a long time for someone to film',
    out_type=int
))

['▁I', '▁have', '▁wa', 'ited', '▁a', '▁long', '▁time', '▁for', '▁someone', '▁to', '▁film']
[41, 142, 1364, 1121, 4, 668, 285, 93, 1079, 33, 91]


In [18]:
# 네이버 영화리뷰 토큰화
import pandas as pd
import sentencepiece as spm
import urllib.request
import csv

In [19]:
# 데이터 다운
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings.txt", filename="ratings.txt")

('ratings.txt', <http.client.HTTPMessage at 0x29fa9161300>)

In [20]:
naver_df = pd.read_table('ratings.txt')
naver_df[:5]

,id,document,label
0,8112052,어릴때보고 지금다시봐도 재밌어요ㅋㅋ,1
1,8132799,"디자인을 배우는 학생으로, 외국디자이너와 그들이 일군 전통을 통해 발전해가는 문화산...",1
2,4655635,폴리스스토리 시리즈는 1부터 뉴까지 버릴께 하나도 없음.. 최고.,1
3,9251303,와.. 연기가 진짜 개쩔구나.. 지루할거라고 생각했는데 몰입해서 봤다.. 그래 이런...,1
4,10067386,안개 자욱한 밤하늘에 떠 있는 초승달 같은 영화.,1


In [21]:
print('리뷰 개수  :',len(naver_df)) # 리뷰 개수 출력

리뷰 개수  : 200000


In [22]:
# Null 값이 존재하므로 이를 제거한 후에 수행
print(naver_df.isnull().values.any())

True


In [23]:
# Null 값이 존재하는 행 제거
naver_df = naver_df.dropna(how='any')

# Null 값이 존재하는지 확인
print(naver_df.isnull().values.any())

False


In [24]:
# 리뷰개수 출력
print('리뷰 개수  :',len(naver_df)) 

리뷰 개수  : 199992


In [25]:
# 네이버 리뷰 데이터를 텍스트 파일로 저장
with open('naver_review.txt', 'w', encoding='utf8') as f:
    # 각 리뷰(document)를 줄바꿈(\n)으로 연결하여 파일에 작성
    f.write('\n'.join(naver_df['document']))

In [26]:
# SentencePiece BPE 토크나이저 학습
spm.SentencePieceTrainer.Train(
    '--input=naver_review.txt '        # 학습에 사용할 텍스트 데이터
    '--model_prefix=naver '            # 생성될 모델 파일 이름 (naver.model, naver.vocab)
    '--vocab_size=5000 '               # 서브워드 vocabulary 크기
    '--model_type=bpe '                # BPE(Byte Pair Encoding) 방식 사용
    '--max_sentence_length=9999'       # 학습에 사용할 최대 문장 길이
)

In [27]:
# SentencePiece에서 생성된 vocab 파일 읽기
vocab_list = pd.read_csv(
    'naver.vocab',
    sep='\t',            # 탭 기준으로 분리
    header=None,         # 헤더 없음
    quoting=csv.QUOTE_NONE
)

# 상위 10개 토큰 확인
vocab_list[:10]

,0,1
0,<unk>,0
1,<s>,0
2,</s>,0
3,..,0
4,영화,-1
5,▁영화,-2
6,▁이,-3
7,▁아,-4
8,...,-5
9,ᄏᄏ,-6


In [28]:
vocab_list.sample(10)

,0,1
3070,이후,-3067
196,하지,-193
4268,덴,-4265
4068,푸,-4065
2618,디어,-2615
3629,외,-3626
4811,빰,-4808
4011,륜,-4008
456,었던,-453
4895,넛,-4892


In [29]:
len(vocab_list)

5000

In [30]:
# 학습된 토크나이저 모델 불러오기
# SentencePiece 토크나이저 객체 생성
sp = spm.SentencePieceProcessor()

# 학습된 SentencePiece 모델 파일 지정
vocab_file = "naver.model"

# 모델 로드
sp.load(vocab_file)

True

In [31]:
sp = spm.SentencePieceProcessor() 
vocab_file = "naver.model" 
sp.load(vocab_file)

True

In [32]:
# 테스트할 문장 리스트
lines = [
    "뭐   이 딴   것 도   영 화 냐 .",
    "진 짜   최 고 의   영 화 입 니 다   ㅋ ㅋ ",
]

# 각 문장에 대해 SentencePiece 토큰화 결과 확인
for line in lines:
    print(line)  # 원문 출력

    # 문장을 서브워드(subword) 토큰으로 분리
    print(sp.encode_as_pieces(line))

    # 서브워드 토큰을 정수 ID로 변환
    print(sp.encode_as_ids(line))

    # 문장 구분을 위한 빈 줄 출력
    print()

뭐   이 딴   것 도   영 화 냐 .
['▁뭐', '▁이', '▁딴', '▁것', '▁도', '▁영', '▁화', '▁', '냐', '▁.']
[136, 6, 2496, 109, 225, 162, 284, 3275, 3487, 405]

진 짜   최 고 의   영 화 입 니 다   ㅋ ㅋ 
['▁진', '▁짜', '▁최', '▁고', '▁의', '▁영', '▁화', '▁입', '▁니', '▁다', '▁ᄏ', '▁ᄏ']
[26, 275, 37, 150, 221, 162, 284, 916, 904, 19, 451, 451]



In [33]:
# 단어집합 크기
sp.GetPieceSize()

5000

In [34]:
# 정수로 부터 맵핑되는 서브워드 로 변환
sp.IdToPiece(4)

'영화'

In [35]:
# PieceToId : 서브워드로부터 맵핑되는 정수로 변환.
sp.PieceToId('영화')

4

In [36]:
# DecodeIds : 정수 시퀀스로부터 문장으로 변환
sp.DecodeIds([54, 200, 821, 85])

'진짜 원 산~~'

In [37]:
# DecodePieces : 서브워드 시퀀스로부터 문장으로 변환
sp.DecodePieces(['▁진 짜 ', '▁최 고 의 ', '▁영 화 입 니 다 ', '▁ᄏ ᄏ '])

'▁진 짜 ▁최 고 의 ▁영 화 입 니 다 ▁ᄏ ᄏ '

In [38]:
# encode : 문장으로부터 인자값에 따라서 정수 시퀀스 또는 서브워드 시퀀스로 변환 가능

print(sp.encode('진 짜 최 고 의 영 화 입 니 다 ㅋ ㅋ ', out_type=str))
print(sp.encode('진 짜 최 고 의 영 화 입 니 다 ㅋ ㅋ ', out_type=int))

['▁진', '▁짜', '▁최', '▁고', '▁의', '▁영', '▁화', '▁입', '▁니', '▁다', '▁ᄏ', '▁ᄏ']
[26, 275, 37, 150, 221, 162, 284, 916, 904, 19, 451, 451]


# 서브워드 텍스트 인코더
텐서플로우를 통해 사용할 수 있는 서브워드 토크나이저입니다. BPE 와 유사한 알고리즘인 Wordpiece Model 을 채택.
Wordpiece Model 을 채택하였으며, 패키지를 통해 쉽게 단어들을 서브워드들로 분리할 수 있다.

In [39]:
!pip install tensorflow-datasets


In [40]:
# 환경문제로 실행 안됨 다운그레이드. 
!pip install protobuf==3.20.3

  Using cached protobuf-3.20.3-cp310-cp310-win_amd64.whl.metadata (698 bytes)
Using cached protobuf-3.20.3-cp310-cp310-win_amd64.whl (904 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.32.0
    Uninstalling protobuf-6.32.0:
      Successfully uninstalled protobuf-6.32.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.40.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.3 requires protobuf<=6.32,>=4.21.6; python_version < "3.11", but you have protobuf 3.20.3 which is incompatible.


In [41]:
import pandas as pd
import urllib.request


In [ ]:
# 버전 통일
# !pip uninstall tensorflow tensorflow-datasets protobuf -y
# !pip install tensorflow==2.12.0
# !pip install tensorflow-datasets==4.4.0
# !pip install protobuf==3.20.3

In [43]:
import tensorflow_datasets as tfds

c:\AI\envs\ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [44]:
# IMDb 리뷰 데이터 CSV 파일 다운로드
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/LawrenceDuan/IMDb-Review-Analysis/master/IMDb_Reviews.csv",
    filename="IMDb_Reviews.csv"
)

# 다운로드한 CSV 파일을 pandas DataFrame으로 불러오기
train_df = pd.read_csv('IMDb_Reviews.csv')

In [45]:
# ’review’ 에 해당하는 열이 토큰화를 수행해야 할 데이터인지 확인
train_df['review']

0        My family and I normally do not watch local mo...
1        Believe it or not, this was at one time the wo...
2        After some internet surfing, I found the "Home...
3        One of the most unheralded great works of anim...
4        It was the Sixties, and anyone with long hair ...
                               ...                        
49995    the people who came up with this are SICK AND ...
49996    The script is so so laughable... this in turn,...
49997    "So there's this bride, you see, and she gets ...
49998    Your mind will not be satisfied by this nobud...
49999    The chaser's war on everything is a weekly sho...
Name: review, Length: 50000, dtype: object

In [46]:
import sentencepiece as spm

spm.SentencePieceTrainer.Train(
    '--input=imdb_review.txt '
    '--model_prefix=imdb '
    '--vocab_size=8000 '
    '--model_type=bpe'
)

In [47]:
# !pip install tensorflow-datasets==4.4.0

In [48]:
# SubwordTextEncoder 직접 임포트
try:
    from tensorflow_datasets.features.text import SubwordTextEncoder
except ImportError:
    try:
        from tensorflow_datasets.core.features.text import SubwordTextEncoder
    except ImportError:
        from tensorflow_datasets.core.deprecated.text import SubwordTextEncoder


In [49]:
!pip install tensorflow-datasets==3.2.1

  Using cached protobuf-6.32.0-cp310-abi3-win_amd64.whl.metadata (593 bytes)
Using cached protobuf-6.32.0-cp310-abi3-win_amd64.whl (435 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.12.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 6.32.0 which is incompatible.


In [50]:
from tensorflow_datasets.features.text import SubwordTextEncoder

tokenizer = SubwordTextEncoder.build_from_corpus(
    train_df['review'],
    target_vocab_size=2**13
)

ModuleNotFoundError: No module named 'tensorflow_datasets.features'

In [ ]:
print(tokenizer.subwords[:100])

NameError: name 'tokenizer' is not defined

In [ ]:
# SubwordTextEncoder는 tfds 버전 충돌로 생략
